<a href="https://colab.research.google.com/github/sooyeon904/kfa-ai-agent/blob/main/KFA_2%EC%A3%BC%EC%B0%A8_%EC%82%AC%EC%A0%84%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KFA AI 에이전트 스터디 사전과제

## 리포트 데이터 소스 찾기

### 선정 리포트

- 증권사: 하나증권
- 리포트 제목: 증권 - 3Q26 Pre, 실적보다는 배당에 초점을
- 발행일: 2026년 10월 14일
- 리포트 링크: [하나증권 증권 산업 리포트 보기](https://stock.naver.com/research/industry/46469)

### 선정 도표

1. 도표 3: 삼성증권 2026년 3분기 연결 실적 예상치
2. 도표 5: 키움증권 2026년 3분기 연결 실적 예상치

두 도표에는 실제 실적과 하나증권의 예상치가 함께 포함되어 있다.
이 과제에서는 실제 실적 중 2026년 2분기 연결 영업이익과 지배주주순이익의 원천자료를 확인하고, Open DART API를 통한 수집을 시도한다.

3분기와 4분기 예상치는 하나증권의 추정치로, DART에서 수집하는 실제 공시 실적과 구분한다.

In [2]:
from google.colab import userdata

try:
    dart_key = userdata.get("DART_API_KEY")
    if dart_key:
        print("DART 키 확인 완료!")
    else:
        print("DART 키가 비어 있어요.")
except Exception:
    print("DART 키를 불러오지 못했어요.")
    print("왼쪽 열쇠 아이콘에서 키 이름과 노트북 액세스를 확인해 주세요.")

DART 키 확인 완료!


In [4]:
import requests

# DART에서 사용하는 회사 고유번호
회사목록 = {
    "삼성증권": "00104856",
    "키움증권": "00296290"
}

주소 = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

for 회사명, 고유번호 in 회사목록.items():
    print(f"\n[{회사명}] 조회 중...")

    try:
        응답 = requests.get(
            주소,
            params={
                "crtfc_key": dart_key,
                "corp_code": 고유번호,
                "bsns_year": "2026",
                "reprt_code": "11012",  # 반기보고서
                "fs_div": "CFS"         # 연결 재무제표
            },
            timeout=(10, 20)
        )

        데이터 = 응답.json()

        if 데이터.get("status") == "000":
            print("조회 성공!")
            print("재무제표 항목 수:", len(데이터.get("list", [])))
        else:
            print("조회 결과:", 데이터.get("status"), 데이터.get("message"))

    except requests.exceptions.RequestException:
        print("DART 연결이 지연되거나 실패했어요.")
    except ValueError:
        print("DART의 응답을 읽지 못했어요.")


[삼성증권] 조회 중...
조회 성공!
재무제표 항목 수: 192

[키움증권] 조회 중...
조회 성공!
재무제표 항목 수: 309


In [5]:
# 두 회사의 연결 포괄손익계산서에서 필요한 항목을 찾습니다.
for 회사명, 고유번호 in 회사목록.items():
    print(f"\n[{회사명}]")

    try:
        응답 = requests.get(
            주소,
            params={
                "crtfc_key": dart_key,
                "corp_code": 고유번호,
                "bsns_year": "2026",
                "reprt_code": "11012",
                "fs_div": "CFS"
            },
            timeout=(10, 20)
        )

        데이터 = 응답.json()

        if 데이터.get("status") != "000":
            print("조회 결과:", 데이터.get("message"))
            continue

        찾은항목 = False

        for 항목 in 데이터.get("list", []):
            이름 = 항목.get("account_nm", "")
            재무제표 = 항목.get("sj_div", "")

            if 재무제표 in ["IS", "CIS"] and (
                이름 == "영업이익"
                or ("지배기업" in 이름 and "순이익" in 이름)
            ):
                찾은항목 = True
                print("항목:", 이름)
                print("  당기 금액:", 항목.get("thstrm_amount", "없음"))
                print("  당기 누적 금액:", 항목.get("thstrm_add_amount", "없음"))

        if not 찾은항목:
            print("조건에 맞는 항목을 찾지 못했어요. 항목 이름을 다시 확인해야 해요.")

    except requests.exceptions.RequestException:
        print("DART 연결이 지연되거나 실패했어요.")
    except ValueError:
        print("DART의 응답을 읽지 못했어요.")


[삼성증권]
항목: 지배기업소유주지분순이익
  당기 금액: 488224283447
  당기 누적 금액: 939080352543
항목: 영업이익
  당기 금액: 675794814040
  당기 누적 금액: 1285326557425

[키움증권]
항목: 지배기업의 소유주에게 귀속되는 반기순이익(손실)
  당기 금액: 680072348384
  당기 누적 금액: 1156435569844
항목: 영업이익
  당기 금액: 788949832477
  당기 누적 금액: 1410170449849


## 데이터 원천 출처 정리

| 차트·표 내용 | 원천 출처 | 링크 | 기준 날짜 | 갱신 주기 | 무료/유료 | 수집 방법 |
|---|---|---|---|---|---|---|
| 도표 3: 삼성증권 연결 실적 중 영업이익·지배주주순이익 | 삼성증권 반기보고서, 연결 포괄손익계산서 | [DART 반기보고서](https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260814002836) | 2026년 2분기 | 분기별 | 무료 | ① Open DART 공식 API |
| 도표 5: 키움증권 연결 실적 중 영업이익·지배주주순이익 | 키움증권 반기보고서, 연결 포괄손익계산서 | [DART 반기보고서](https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260813001767) | 2026년 2분기 | 분기별 | 무료 | ① Open DART 공식 API |

### 확인 결과

각 회사의 2026년 2분기 연결 영업이익과 지배주주순이익을
반기보고서 및 Open DART API 조회 결과와 대조하였다.

리포트의 3분기·4분기 예상치는 하나증권의 추정치이며,
위 표에서 조회한 실제 공시 실적과 구분한다.

## 검증 범위

리서치의 도표 3과 도표 5에 표시된 2026년 2분기 실제 실적 중
연결 영업이익과 지배주주순이익을 확인하였다.

각 회사의 반기보고서와 Open DART API 조회 결과를 대조한 결과,
아래 네 항목이 리서치에 표시된 수치와 일치했다.

| 회사 | 항목 | DART 조회 금액 | 리서치 수치 | 결과 |
|---|---|---:|---:|---|
| 삼성증권 | 연결 영업이익 | 675,794,814,040원 | 675.8십억 원 | 일치 |
| 삼성증권 | 지배주주순이익 | 488,224,283,447원 | 488.2십억 원 | 일치 |
| 키움증권 | 연결 영업이익 | 788,949,832,477원 | 788.9십억 원 | 일치 |
| 키움증권 | 지배주주순이익 | 680,072,348,384원 | 680.1십억 원 | 일치 |

리서치는 십억 원 단위로 소수점 첫째 자리까지 표시한다.
API의 당기 금액은 2분기 3개월 실적이며, 당기 누적 금액은
1~2분기 상반기 누적 실적이다.

도표의 모든 수치와 예상치를 검증한 것은 아니다.
3분기와 4분기 예상치는 하나증권의 추정치로, 공시된 실제 실적과 구분한다.

## 데이터 수집 방법

두 도표에서 확인한 실제 실적은 Open DART 공식 API를 통해 수집했다.

- API: Open DART 단일회사 전체 재무제표
- 수집 대상: 삼성증권, 키움증권
- 사업연도: 2026년
- 보고서: 반기보고서
- 재무제표: 연결재무제표
- 확인 항목: 영업이익, 지배주주순이익
- 기준 기간: 2026년 2분기 3개월 및 상반기 누적
- 단위: 원

### 수집 순서

1. 각 회사의 DART 고유번호를 확인한다.
2. 사업연도, 보고서 종류, 연결재무제표를 지정해 API를 호출한다.
3. 반환된 항목 중 영업이익과 지배주주순이익을 찾는다.
4. 2분기 금액을 반기보고서 및 리서치의 실제 실적과 비교한다.

※ 리서치의 예상치는 하나증권의 추정치로, DART 공시 실적과 구분한다.

In [6]:
import requests
import pandas as pd
from google.colab import userdata

# 코랩에 저장된 DART 키 사용
dart_key = userdata.get("DART_API_KEY")

# 조사할 회사와 보고서
회사목록 = {
    "삼성증권": "00104856",
    "키움증권": "00296290"
}

기간목록 = [
    ("2025년 3분기", "2025", "11014"),
    ("2026년 2분기", "2026", "11012")
]

주소 = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
결과 = []

for 회사명, 고유번호 in 회사목록.items():
    for 기간, 연도, 보고서코드 in 기간목록:
        print(f"{회사명} {기간} 조회 중...")

        try:
            응답 = requests.get(
                주소,
                params={
                    "crtfc_key": dart_key,
                    "corp_code": 고유번호,
                    "bsns_year": 연도,
                    "reprt_code": 보고서코드,
                    "fs_div": "CFS"
                },
                timeout=(10, 20)
            )

            데이터 = 응답.json()

            if 데이터.get("status") != "000":
                print("조회 실패:", 데이터.get("message"))
                continue

            for 항목 in 데이터.get("list", []):
                이름 = 항목.get("account_nm", "")
                재무제표 = 항목.get("sj_div", "")

                if 재무제표 in ["IS", "CIS"] and (
                    이름 == "영업이익"
                    or ("지배기업" in 이름 and "순이익" in 이름)
                ):
                    결과.append({
                        "회사": 회사명,
                        "기간": 기간,
                        "항목": 이름,
                        "해당 분기 금액(원)": 항목.get("thstrm_amount", ""),
                        "누적 금액(원)": 항목.get("thstrm_add_amount", "")
                    })

        except (requests.exceptions.RequestException, ValueError):
            print("조회 중 연결 또는 응답 오류가 발생했어요.")

if 결과:
    print("\n조회 결과")
    display(pd.DataFrame(결과))
else:
    print("\n조회된 결과가 없어요. 오류 메시지를 확인해 주세요.")

삼성증권 2025년 3분기 조회 중...
삼성증권 2026년 2분기 조회 중...
키움증권 2025년 3분기 조회 중...
키움증권 2026년 2분기 조회 중...

조회 결과


,회사,기간,항목,해당 분기 금액(원),누적 금액(원)
0,삼성증권,2025년 3분기,지배기업소유주지분순이익,309174366337,792231898731
1,삼성증권,2025년 3분기,영업이익,401812408978,1045106742027
2,삼성증권,2026년 2분기,지배기업소유주지분순이익,488224283447,939080352543
3,삼성증권,2026년 2분기,영업이익,675794814040,1285326557425
4,키움증권,2025년 3분기,영업이익,408881346062,1142633564981
5,키움증권,2026년 2분기,지배기업의 소유주에게 귀속되는 반기순이익(손실),680072348384,1156435569844
6,키움증권,2026년 2분기,영업이익,788949832477,1410170449849


In [7]:
# 키움증권 2025년 3분기의 이익 관련 항목 이름 확인
응답 = requests.get(
    주소,
    params={
        "crtfc_key": dart_key,
        "corp_code": "00296290",
        "bsns_year": "2025",
        "reprt_code": "11014",
        "fs_div": "CFS"
    },
    timeout=(10, 20)
)

데이터 = 응답.json()

if 데이터.get("status") == "000":
    for 항목 in 데이터.get("list", []):
        이름 = 항목.get("account_nm", "")
        재무제표 = 항목.get("sj_div", "")

        if 재무제표 in ["IS", "CIS"] and (
            "순이익" in 이름 or "지배" in 이름
        ):
            print(
                이름,
                "| 해당 분기:", 항목.get("thstrm_amount", ""),
                "| 누적:", 항목.get("thstrm_add_amount", "")
            )
else:
    print("조회 결과:", 데이터.get("status"), 데이터.get("message"))

비지배주주지분 | 해당 분기: 475679726 | 누적: 1145842087
지배주주지분 | 해당 분기: 400184585389 | 누적: 1075716528767
당기순이익 | 해당 분기: 322382473016 | 누적: 868115199969
비지배주주지분 | 해당 분기: 472386850 | 누적: 1164679052
지배주주지분 | 해당 분기: 321910086166 | 누적: 866950520917
법인세비용차감전순이익 | 해당 분기: 439213389866 | 누적: 1181716630653
